In [10]:

# Experiment 6: Golden Question Set + Baseline Run -> Release v0.1

!pip -q install pymongo google-genai pandas

import time
import json
import datetime
import pandas as pd

from google.colab import userdata
from google import genai
from pymongo import MongoClient

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

MODEL = "gemini-2.5-flash"
RETRIES = 4

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
MONGODB_URI = userdata.get("MONGODB_URI")

client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options={"timeout": 60000}
)

mongo_client = MongoClient(
    MONGODB_URI,
    serverSelectionTimeoutMS=15000
)

mongo_client.admin.command("ping")

db = mongo_client["genai_course"]

print("Gemini initialized")
print("MongoDB connected successfully")

# --------------------------------------------------
# 2. GOLDEN QUESTION SET (ALL 32 QUESTIONS)
# --------------------------------------------------

GOLDEN = [
    ("What does BPE stand for?", ["byte", "pair"], "Module I"),
    ("Which tokenizer does BERT use?", ["wordpiece"], "Module I"),
    ("Which tokenizer family do T5 and mT5 use?", ["sentencepiece"], "Module I"),
    ("Why does Telugu text usually cost more tokens than English?", ["token"], "Module I"),
    ("What is a knowledge cutoff of an LLM?", ["date"], "Module I"),
    ("What is a hallucination in an LLM?", ["false"], "Module I"),
    ("What is next-token prediction?", ["next", "token"], "Module I"),
    ("What is a context window?", ["token"], "Module I"),

    ("Name the three vectors used in attention.", ["query", "key", "value"], "Module II"),
    ("Which function turns attention scores into weights?", ["softmax"], "Module II"),
    ("Is GPT encoder-only or decoder-only?", ["decoder"], "Module II"),
    ("Is BERT encoder-only or decoder-only?", ["encoder"], "Module II"),
    ("What kind of architecture is T5?", ["encoder", "decoder"], "Module II"),
    ("What does a higher temperature do during sampling?", ["random"], "Module II"),
    ("What is top-k sampling?", ["k"], "Module II"),
    ("What is top-p sampling also called?", ["nucleus"], "Module II"),

    ("What does TF-IDF stand for?", ["term frequency", "inverse document frequency"], "Module III"),
    ("Name a classic sparse ranking function used in search engines.", ["bm25"], "Module III"),
    ("What does HNSW stand for?", ["hierarchical", "navigable", "small world"], "Module III"),
    ("What does RAG stand for?", ["retrieval", "augmented", "generation"], "Module III"),
    ("What does MRR stand for in retrieval evaluation?", ["mean", "reciprocal", "rank"], "Module III"),
    ("Why do we add overlap when chunking documents?", ["context"], "Module III"),

    ("What is few-shot prompting?", ["example"], "Module IV"),
    ("Which Python library is used for schema-validated structured outputs in this course?", ["pydantic"], "Module IV"),
    ("What is prompt injection?", ["instruction"], "Module IV"),

    ("What does LoRA stand for?", ["low", "rank", "adaptation"], "Module V"),
    ("What does the Q in QLoRA refer to?", ["quantiz"], "Module V"),
    ("What does DPO stand for?", ["direct", "preference", "optimization"], "Module V"),
    ("What does RLHF stand for?", ["reinforcement", "human", "feedback"], "Module V"),

    ("Which Python framework is used for the microservices in this course?", ["fastapi"], "Module VI"),
    ("Which tool is used for Infrastructure as Code on AWS in this course?", ["terraform"], "Module VI"),
    ("What does MCP stand for?", ["model", "context", "protocol"], "Module VI"),
]

golden = [
    {
        "id": f"Q{i+1:02d}",
        "question": q,
        "expected_keywords": k,
        "source": s
    }
    for i, (q, k, s) in enumerate(GOLDEN)
]

# --------------------------------------------------
# 3. SAVE GOLDEN SET
# --------------------------------------------------

with open("golden_set.json", "w") as f:
    json.dump(golden, f, indent=2)

db.golden_set.delete_many({})
db.golden_set.insert_many([dict(g) for g in golden])

print(
    "Golden set:",
    len(golden),
    "questions saved to golden_set.json and MongoDB"
)

# --------------------------------------------------
# 4. GEMINI API WITH RETRY HANDLING
# --------------------------------------------------

def ask(prompt, retries=RETRIES):

    for attempt in range(retries):

        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt
            )

            return response

        except Exception as e:

            error = str(e)
            print(f"Retry {attempt + 1} -> {error[:250]}")

            # Do not retry permanent permission errors
            if "403" in error or "PERMISSION_DENIED" in error:
                raise RuntimeError(
                    "Gemini API permission denied. "
                    "Check API key and Google AI Studio project access. "
                    + error[:500]
                )

            if attempt < retries - 1:
                time.sleep(5 * (2 ** attempt))

    raise RuntimeError("LLM call failed after retries")

# --------------------------------------------------
# 5. ANSWER SCORING
# --------------------------------------------------

def score(answer, keywords):

    answer = answer.lower().replace("-", " ")

    return all(
        keyword.lower() in answer
        for keyword in keywords
    )

# --------------------------------------------------
# 6. BASELINE EXPERIMENT
# --------------------------------------------------

results = []

for index, g in enumerate(golden):

    print(f"\nProcessing {g['id']}: {g['question']}")

    t = time.time()

    try:

        r = ask(
            "Answer in one or two sentences: " + g["question"]
        )

        ans = (r.text or "").strip()

        usage = r.usage_metadata

        input_tokens = usage.prompt_token_count or 0
        output_tokens = usage.candidates_token_count or 0

        results.append({
            "id": g["id"],
            "source": g["source"],
            "correct": score(ans, g["expected_keywords"]),
            "latency_s": round(time.time() - t, 2),
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "answer": ans[:90]
        })

        print("Answer:", ans[:150])
        print("Correct:", score(ans, g["expected_keywords"]))

    except Exception as e:

        print("FAILED:", str(e)[:300])

        results.append({
            "id": g["id"],
            "source": g["source"],
            "correct": False,
            "latency_s": round(time.time() - t, 2),
            "input_tokens": 0,
            "output_tokens": 0,
            "answer": "ERROR: " + str(e)[:90]
        })

        if "permission denied" in str(e).lower():
            print("Stopping experiment because of API permissions.")
            break

    time.sleep(4)

# --------------------------------------------------
# 7. RESULTS AND BASELINE
# --------------------------------------------------

df = pd.DataFrame(results)

print("\nBASELINE RESULTS")

print(
    df[["id", "source", "correct", "latency_s", "answer"]]
    .to_string(index=False)
)

if len(df) > 0:

    summary = {
        "release": "v0.1",
        "model": MODEL,
        "date": str(datetime.date.today()),
        "questions": len(df),
        "completed": len(df),
        "accuracy": round(df.correct.mean(), 3),
        "avg_latency_s": round(df.latency_s.mean(), 2),
        "avg_input_tokens": round(df.input_tokens.mean(), 1),
        "avg_output_tokens": round(df.output_tokens.mean(), 1)
    }

    # Save release summary
    db.releases.insert_one(dict(summary))

    # Save CSV
    df.to_csv("baseline_v0.1.csv", index=False)

    # Save JSON results
    df.to_json("baseline_v0.1.json", orient="records", indent=2)

    print("\nRelease v0.1 baseline:")
    print(summary)

    print("\nModule-wise accuracy:")
    print(df.groupby("source").correct.mean().round(2))

    print("\nFiles generated:")
    print("1. golden_set.json")
    print("2. baseline_v0.1.csv")
    print("3. baseline_v0.1.json")

else:
    print("No baseline results generated.")


Gemini initialized
MongoDB connected successfully
Golden set: 32 questions saved to golden_set.json and MongoDB

Processing Q01: What does BPE stand for?
Retry 1 -> 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'The Gemini API free tier is not available for this project. To use the Gemini API, please set up billing to upgrade to the paid tier at https://aistudio.google.com/billing. For more informati
FAILED: Gemini API permission denied. Check API key and Google AI Studio project access. 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'The Gemini API free tier is not available for this project. To use the Gemini API, please set up billing to upgrade to the paid tier at https://aistudio.google.
Stopping experiment because of API permissions.

BASELINE RESULTS
 id   source  correct  latency_s                                                                                            answer
Q01 Module I    False       0.17 ERROR: Gemini API permission denied. Check AP